## 1. Setup

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# API configuration
API_URL = os.environ.get('API_URL', 'http://api:8000')
print(f"API URL: {API_URL}")

In [ ]:
# Health check
response = requests.get(f"{API_URL}/health", timeout=10)
health = response.json()
print(f"Status: {health['status']}")
print(f"Model trained: {health['model_trained']}")

if not health['model_trained']:
    raise Exception("Model not ready. Run: docker compose up")

## 2. Load Data

In [ ]:
# Fetch museums from API
response = requests.get(f"{API_URL}/museums", timeout=30)
museums_data = response.json()

# Convert to DataFrame
records = []
for m in museums_data['museums']:
    records.append({
        'museum': m['name'],
        'city': m['city']['name'],
        'country': m['city']['country'],
        'visitors': m['visitors'],
        'population': m['city']['population']
    })
df = pd.DataFrame(records)
print(f"Loaded {len(df)} museums")
df.head(10)

## 3. Model Statistics

In [ ]:
# Get model statistics
response = requests.get(f"{API_URL}/regression/stats", timeout=10)
stats = response.json()

print("Regression Model Statistics")
print("=" * 40)
print(f"R² Score:    {stats['r2_score']:.4f}")
print(f"RMSE:        {stats['rmse']:,.0f}")
print(f"MAE:         {stats['mae']:,.0f}")
print(f"Samples:     {stats['n_samples']}")
print(f"\nEquation: {stats['equation']}")

## 4. Visualization

In [ ]:
# Scatter plot with regression line
fig, ax = plt.subplots(figsize=(10, 6))

# Convert to millions
x = df['population'] / 1_000_000
y = df['visitors'] / 1_000_000

ax.scatter(x, y, alpha=0.7, s=80, edgecolors='black', linewidth=0.5)

# Regression line
x_line = np.linspace(x.min(), x.max(), 100)
y_line = (stats['coefficient'] * x_line * 1_000_000 + stats['intercept']) / 1_000_000
ax.plot(x_line, y_line, 'r-', linewidth=2, label=f"R² = {stats['r2_score']:.3f}")

ax.set_xlabel('City Population (millions)')
ax.set_ylabel('Annual Visitors (millions)')
ax.set_title('Museum Visitors vs City Population')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Predictions

In [ ]:
# Test predictions
test_populations = [1_000_000, 5_000_000, 10_000_000, 20_000_000]

print("Visitor Predictions")
print("=" * 40)
print(f"{'Population':>15} | {'Predicted Visitors':>20}")
print("-" * 40)

for pop in test_populations:
    response = requests.get(f"{API_URL}/regression/predict", params={'population': pop}, timeout=10)
    pred = response.json()
    print(f"{pred['population']:>15,} | {pred['predicted_visitors']:>20,}")

## 6. Interpretation

The linear regression model shows the relationship between city population and museum visitor attendance.

**Key findings:**
- The coefficient indicates how many additional visitors are expected per unit increase in population
- R² score shows the proportion of variance explained by the model
- This simple model demonstrates the core concept; production systems would include additional features

**Limitations:**
- Linear model assumes a linear relationship
- Does not account for tourism factors, museum reputation, or economic conditions
- Based on Wikipedia data which may have inconsistencies